In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🚪 API Gateway & Rate Limiting Architecture Guide

**Building secure, scalable API gateways with rate limiting and traffic management**

This guide covers:
- API Gateway patterns (AWS API Gateway, Kong, Nginx)
- Rate limiting strategies
- Authentication & authorization
- Request/response transformation
- API versioning
- Throttling and quota management
- Monitoring and metrics
- Cost optimization
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. API Gateway Fundamentals

### AWS API Gateway Architecture
```python
# Using Python boto3
import boto3
import json

apigateway = boto3.client('apigateway')

# Create REST API
api = apigateway.create_rest_api(
    name='my-api',
    description='My application API',
    endpointConfiguration={
        'types': ['REGIONAL']
    }
)

api_id = api['id']

# Get root resource
resources = apigateway.get_resources(restApiId=api_id)
root_id = resources['items'][0]['id']

# Create /users resource
users_resource = apigateway.create_resource(
    restApiId=api_id,
    parentId=root_id,
    pathPart='users'
)

users_resource_id = users_resource['id']

# Create GET method
apigateway.put_method(
    restApiId=api_id,
    resourceId=users_resource_id,
    httpMethod='GET',
    authorizationType='AWS_IAM'  # or 'CUSTOM', 'COGNITO_USER_POOLS'
)

# Set up Lambda integration
apigateway.put_integration(
    restApiId=api_id,
    resourceId=users_resource_id,
    httpMethod='GET',
    type='AWS_PROXY',
    integrationHttpMethod='POST',
    uri=f'arn:aws:apigateway:us-east-1:lambda:path/2015-03-31/functions/arn:aws:lambda:us-east-1:123456789012:function:get-users/invocations'
)

# Deploy to stage
deployment = apigateway.create_deployment(
    restApiId=api_id,
    stageName='prod'
)

print(f"API deployed: https://{api_id}.execute-api.us-east-1.amazonaws.com/prod")
```

### HTTP/REST vs GraphQL Gateways
```
REST Gateway:
├─ Path-based routing
├─ HTTP method routing (GET, POST, PUT, DELETE)
├─ Header-based authentication
├─ Request/response transformation
└─ Rate limiting per endpoint

GraphQL Gateway:
├─ Single endpoint (usually /graphql)
├─ Query complexity analysis
├─ Depth limiting
├─ Query cost calculation
└─ More granular rate limiting (per query operation)
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Rate Limiting Strategies

### Token Bucket Algorithm
```python
from datetime import datetime, timedelta
from collections import defaultdict
import asyncio

class TokenBucketLimiter:
    def __init__(self, capacity: int, refill_rate: float):
        """
        Args:
            capacity: Max tokens in bucket
            refill_rate: Tokens per second
        """
        self.capacity = capacity
        self.refill_rate = refill_rate
        self.buckets = defaultdict(lambda: {
            'tokens': capacity,
            'last_refill': datetime.now()
        })
    
    def is_allowed(self, client_id: str) -> bool:
        """Check if request is allowed"""
        bucket = self.buckets[client_id]
        
        # Calculate elapsed time
        now = datetime.now()
        elapsed = (now - bucket['last_refill']).total_seconds()
        
        # Add tokens based on elapsed time
        bucket['tokens'] = min(
            self.capacity,
            bucket['tokens'] + elapsed * self.refill_rate
        )
        bucket['last_refill'] = now
        
        # Check if tokens available
        if bucket['tokens'] >= 1:
            bucket['tokens'] -= 1
            return True
        
        return False

# Usage
limiter = TokenBucketLimiter(capacity=100, refill_rate=10)  # 100 requests per 10 seconds

if limiter.is_allowed('user_123'):
    print("Request allowed")
else:
    print("Rate limit exceeded")
```

### Sliding Window Algorithm
```python
from datetime import datetime, timedelta
from collections import defaultdict
from collections import deque

class SlidingWindowLimiter:
    def __init__(self, limit: int, window_seconds: int):
        """
        Args:
            limit: Max requests in window
            window_seconds: Time window in seconds
        """
        self.limit = limit
        self.window = timedelta(seconds=window_seconds)
        self.requests = defaultdict(deque)
    
    def is_allowed(self, client_id: str) -> bool:
        """Check if request is allowed"""
        now = datetime.now()
        requests = self.requests[client_id]
        
        # Remove old requests outside window
        while requests and (now - requests[0]) > self.window:
            requests.popleft()
        
        # Check limit
        if len(requests) < self.limit:
            requests.append(now)
            return True
        
        return False

# Usage
limiter = SlidingWindowLimiter(limit=100, window_seconds=60)  # 100 requests per minute
```

### Leaky Bucket Algorithm
```python
from datetime import datetime, timedelta
import asyncio
from collections import defaultdict

class LeakyBucketLimiter:
    def __init__(self, capacity: int, leak_rate: float):
        """
        Args:
            capacity: Bucket size
            leak_rate: Requests per second
        """
        self.capacity = capacity
        self.leak_rate = leak_rate
        self.buckets = defaultdict(lambda: {
            'current': 0,
            'last_leak': datetime.now()
        })
    
    async def is_allowed(self, client_id: str) -> bool:
        """Check if request is allowed"""
        bucket = self.buckets[client_id]
        
        # Leak requests
        now = datetime.now()
        elapsed = (now - bucket['last_leak']).total_seconds()
        leaked = elapsed * self.leak_rate
        
        bucket['current'] = max(0, bucket['current'] - leaked)
        bucket['last_leak'] = now
        
        # Check capacity
        if bucket['current'] < self.capacity:
            bucket['current'] += 1
            return True
        
        return False
```

### Redis-based Distributed Rate Limiting
```python
import redis
import json
from datetime import datetime, timedelta

class RedisRateLimiter:
    def __init__(self, redis_url: str):
        self.redis = redis.from_url(redis_url)
    
    def is_allowed(self, client_id: str, limit: int, window_seconds: int) -> bool:
        """
        Redis-backed rate limiting
        Uses INCR and EXPIRE for atomic operations
        """
        key = f"rate_limit:{client_id}"
        
        # Increment counter
        current = self.redis.incr(key)
        
        # Set expiry on first request
        if current == 1:
            self.redis.expire(key, window_seconds)
        
        # Check limit
        return current <= limit
    
    def get_remaining(self, client_id: str, limit: int) -> int:
        """Get remaining requests"""
        key = f"rate_limit:{client_id}"
        current = int(self.redis.get(key) or 0)
        return max(0, limit - current)

# Usage
limiter = RedisRateLimiter('redis://localhost:6379')

if limiter.is_allowed('user_123', limit=100, window_seconds=60):
    remaining = limiter.get_remaining('user_123', limit=100)
    print(f"Request allowed. Remaining: {remaining}")
else:
    print("Rate limit exceeded")
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. API Gateway Implementation

### Kong API Gateway
```yaml
# kong.yml - Kong gateway configuration
_format_version: "3.0"
_transform: true

services:
  - name: user-service
    url: http://user-api:3000
    routes:
      - name: users-get
        paths:
          - /users
        methods:
          - GET
      - name: users-post
        paths:
          - /users
        methods:
          - POST
    plugins:
      - name: rate-limiting
        config:
          minute: 1000
          policy: redis
          redis_host: redis
      - name: authentication
        config:
          anonymous: ~
      - name: jwt
        config:
          secret: your-secret-key
      - name: cors
        config:
          origins:
            - "http://localhost:3000"
            - "https://example.com"
          credentials: true
      - name: request-transformer
        config:
          add:
            headers:
              - X-Gateway-Version:1.0

  - name: admin-service
    url: http://admin-api:3000
    routes:
      - name: admin-routes
        paths:
          - /admin
        methods:
          - POST
    plugins:
      - name: rate-limiting
        config:
          minute: 100
```

### Nginx API Gateway
```nginx
# nginx.conf - Nginx reverse proxy with rate limiting
http {
    # Rate limiting zones
    limit_req_zone $binary_remote_addr zone=general:10m rate=10r/s;
    limit_req_zone $http_x_api_key zone=api_key:10m rate=100r/s;
    
    upstream user_service {
        server user-api:3000;
    }
    
    upstream admin_service {
        server admin-api:3000;
    }
    
    server {
        listen 8080;
        
        # Public API endpoints
        location /api/users {
            limit_req zone=general burst=20 nodelay;
            
            # CORS headers
            add_header 'Access-Control-Allow-Origin' '*';
            add_header 'Access-Control-Allow-Methods' 'GET, POST, OPTIONS';
            
            proxy_pass http://user_service;
            proxy_set_header X-Real-IP $remote_addr;
            proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        }
        
        # Admin endpoints (stricter rate limit)
        location /admin {
            limit_req zone=api_key burst=10 nodelay;
            
            # Require API key
            if ($http_x_api_key = "") {
                return 401;
            }
            
            proxy_pass http://admin_service;
        }
        
        # Health check endpoint
        location /health {
            access_log off;
            return 200 "healthy\n";
            add_header Content-Type text/plain;
        }
    }
}
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Authentication & Authorization

### JWT-based API Authentication
```python
from fastapi import FastAPI, Depends, HTTPException, status
from fastapi.security import HTTPBearer, HTTPAuthCredentials
import jwt
from datetime import datetime, timedelta

app = FastAPI()
security = HTTPBearer()

SECRET_KEY = "your-secret-key"
ALGORITHM = "HS256"
EXPIRATION_HOURS = 24

def create_access_token(user_id: str, hours: int = EXPIRATION_HOURS):
    """Create JWT token"""
    expires = datetime.utcnow() + timedelta(hours=hours)
    
    payload = {
        'sub': user_id,
        'exp': expires,
        'iat': datetime.utcnow()
    }
    
    token = jwt.encode(payload, SECRET_KEY, algorithm=ALGORITHM)
    return token

async def verify_token(credentials: HTTPAuthCredentials = Depends(security)):
    """Verify JWT token"""
    try:
        payload = jwt.decode(
            credentials.credentials,
            SECRET_KEY,
            algorithms=[ALGORITHM]
        )
        user_id = payload.get('sub')
        
        if user_id is None:
            raise HTTPException(
                status_code=status.HTTP_401_UNAUTHORIZED,
                detail="Invalid token"
            )
        
        return user_id
        
    except jwt.ExpiredSignatureError:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Token expired"
        )
    except jwt.InvalidTokenError:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid token"
        )

@app.post("/login")
async def login(username: str, password: str):
    # Validate credentials (in real app, check database)
    if username == "user" and password == "password":
        token = create_access_token(username)
        return {"access_token": token, "token_type": "bearer"}
    
    raise HTTPException(status_code=401, detail="Invalid credentials")

@app.get("/protected")
async def protected_route(user_id: str = Depends(verify_token)):
    return {"message": f"Hello {user_id}"}
```

### RBAC (Role-Based Access Control)
```python
from enum import Enum
from typing import List

class Role(str, Enum):
    ADMIN = "admin"
    USER = "user"
    GUEST = "guest"

class Permission(str, Enum):
    READ = "read"
    WRITE = "write"
    DELETE = "delete"
    ADMIN = "admin"

# Role to permissions mapping
ROLE_PERMISSIONS = {
    Role.ADMIN: [Permission.READ, Permission.WRITE, Permission.DELETE, Permission.ADMIN],
    Role.USER: [Permission.READ, Permission.WRITE],
    Role.GUEST: [Permission.READ]
}

def require_permission(permission: Permission):
    """Decorator to check permissions"""
    async def permission_checker(user_role: Role = Depends(get_user_role)):
        required_perms = ROLE_PERMISSIONS.get(user_role, [])
        
        if permission not in required_perms:
            raise HTTPException(status_code=403, detail="Insufficient permissions")
        
        return user_role
    
    return permission_checker

@app.delete("/users/{user_id}")
async def delete_user(user_id: str, role = Depends(require_permission(Permission.DELETE))):
    # Delete user logic
    return {"deleted": user_id}
```
</MySCode.Cell>

<MySCode.Cell language="markdown">
## 5. API Gateway Best Practices

✅ **Design**
- [ ] Consistent API versioning (/v1, /v2)
- [ ] Clear and RESTful endpoints
- [ ] Meaningful HTTP status codes
- [ ] Comprehensive error responses

✅ **Security**
- [ ] HTTPS/TLS enforced
- [ ] Authentication on all endpoints
- [ ] CORS properly configured
- [ ] Input validation and sanitization
- [ ] Rate limiting enabled

✅ **Performance**
- [ ] Request/response compression
- [ ] Caching headers set
- [ ] Connection pooling
- [ ] Load balancing across backends

✅ **Monitoring**
- [ ] Request/response logging
- [ ] Performance metrics (latency, throughput)
- [ ] Error rate tracking
- [ ] Alerting configured

✅ **Operations**
- [ ] API documentation current
- [ ] Deprecation notices communicated
- [ ] Rollback procedures documented
- [ ] Health checks working
</MySCode.Cell>
```